# Validation Log and Notebook for AlexNet


The following notebook is going to be used to tweak the alexnet model for the different datasets, this is going to be done in order to make sure that we are picking the right architecture and hyperparameters for the efficiency tradeoff comparison.


##  Parameters Hyperparameters that we are going to change and validate:

Parameters: this are specific to each convulotional layer and focus on modifying how the convulotion operation is applied across the image.
We have to be really careful with this parameter as they dictate how much they are moving.
* Stride: This parameter determines how many units the filter moves at each step.
* Padding: This parameter determines how many zeros are going to be added arounf the input matrix border ir order to retain edge information and avoid shrinking.


Hyperparameters: 

## Vanilla Architecture: AlexNet as it was written on the breakthrough paper.


For this first validation iteration I am going to use the classic alexnet iteration with the vanilla hyperparameter that where established in the AlexNet Paper breakthrough we will see the overall perfomance of this model with the architecture that we have and see if we have to change any of the specific hyperparameters or parameters in the layers.

The first convulotional layer is the one that transforms the input data so thats where we are going to tweak the parameters.
The other hyperparameters in the other layers are going to be fixed.


### Validation with the MNIST dataset.

In [1]:
%pip install torchvision

Note: you may need to restart the kernel to use updated packages.


In [8]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import importlib

import alexnet

importlib.reload(alexnet)
AlexNet4803 = alexnet.AlexNet4803

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose(
    [
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ]
)
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=0)
hyperparameter = [
    {"k": 3, "s": 1, "p": 1, "name": "Small Kernel (3x3), Stride 1"},
    {"k": 5, "s": 1, "p": 2, "name": "Medium Kernel (5x5), Stride 1"},
    {"k": 5, "s": 2, "p": 2, "name": "Medium Kernel (5x5), Stride 2"},
    {"k": 7, "s": 2, "p": 3, "name": "Large Kernel (7x7), Stride 2"},
]
num_epochs = 2
results = {}
def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            pred = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    return 100.0 * correct / total


for config in hyperparameter:
    print(f"\n--- Config: {config['name']} ---")
    model = AlexNet4803(1, 10, k1=config["k"], s1=config["s"], p1=config["p"]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        n_batches = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches += 1
        train_loss = running_loss / max(n_batches, 1)
        test_acc = evaluate(model, test_loader)
        print(f"Epoch {epoch}/{num_epochs} | train_loss={train_loss:.4f} | test_acc={test_acc:.2f}%")

    results[config["name"]] = test_acc
results


--- Config: Small Kernel (3x3), Stride 1 ---


KeyboardInterrupt: 